# Airbnb Pipeline Walkthrough

In this notebook, we'll move from raw Airbnb-style listings and reviews to a small analytics layer enriched by an LLM. The goal is to show how AI fits inside a normal data engineering workflow rather than replacing it.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path('..').resolve()))
listings = pd.read_csv('../data/listings.csv')
reviews = pd.read_csv('../data/reviews.csv')

print('Listings shape:', listings.shape)
print('Reviews shape:', reviews.shape)
display(listings.head())
display(reviews.head())

In [ ]:
from src.clean import clean_listings, clean_reviews

listings_clean = clean_listings(listings)
reviews_clean = clean_reviews(reviews)

print('Before cleaning reviews:', len(reviews))
print('After cleaning reviews:', len(reviews_clean))
display(listings[['name', 'availability_365']].head(3))
display(listings_clean[['name', 'availability_365']].head(3))

In [ ]:
from src.enrich import enrich_reviews
from src.llm import extract_review_insights, get_openai_client

client = get_openai_client()
single_review = reviews_clean.iloc[0]['comments']
display(extract_review_insights(single_review, client))

enriched_reviews = enrich_reviews(reviews_clean, client=client, sample=10)
display(enriched_reviews[['review_id', 'sentiment', 'themes', 'would_recommend', 'summary']].head())

In [ ]:
from src.analytics import build_listing_summary

listing_summary = build_listing_summary(listings_clean, enriched_reviews)
display(listing_summary.head())

In [ ]:
from src.analytics import build_neighbourhood_summary

neighbourhood_summary = build_neighbourhood_summary(listing_summary)
display(neighbourhood_summary.sort_values('average_sentiment_score', ascending=False))

## Adapting to Databricks

The exact same stages map cleanly to Databricks: ingest with Spark, clean review text, enrich comments in a scalable job, and write summary tables to Delta.

```python
listings_spark = spark.read.csv("/dbfs/FileStore/airbnb/listings.csv", header=True, inferSchema=True)
reviews_spark = spark.read.csv("/dbfs/FileStore/airbnb/reviews.csv", header=True, inferSchema=True)
clean_reviews_spark = reviews_spark.filter(F.length(F.trim(F.col("comments"))) > 0)
```
